# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Árboles: no necesitan escalado, solo imputación por mediana
tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), feature_columns),
])

# Modelos basados en distancias o coeficientes: imputación + escalado
scaled_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns),
])

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si el scaler se ajusta antes con todo `X_train`, la media y la desviación estándar incluyen los datos de cada fold de validación. Eso es **fuga de información (data leakage)**: el modelo "ve" algo de los datos con que se evalúa y la métrica de CV queda optimista. Dentro del pipeline, en cada fold el scaler se ajusta solo con la parte de entrenamiento y luego únicamente transforma la de validación, igual que pasará en producción con datos nuevos. Además, el pipeline guardado contiene el preprocesamiento, así que la inferencia aplica exactamente la misma transformación.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("preprocess", clone(tree_preprocessor)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier()),
    ]),
}
models

{'decision_tree': Pipeline(steps=[('preprocess',
                  ColumnTransformer(transformers=[('num',
                                                   SimpleImputer(strategy='median'),
                                                   ['alcohol', 'malic_acid',
                                                    'ash', 'alcalinity_of_ash',
                                                    'magnesium', 'total_phenols',
                                                    'flavanoids',
                                                    'nonflavanoid_phenols',
                                                    'proanthocyanins',
                                                    'color_intensity', 'hue',
                                                    'od280/od315_of_diluted_wines',
                                                    'proline'])])),
                 ('model', DecisionTreeClassifier(random_state=42))]),
 'logistic_regression': Pipeline(steps=[('preprocess',


## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=cv, scoring="accuracy", return_train_score=True,
    )
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "val_accuracy": scores["test_score"].mean(),
        "val_std": scores["test_score"].std(),
        "gap": scores["train_score"].mean() - scores["test_score"].mean(),
    })

initial_comparison = (
    pd.DataFrame(rows).set_index("model")
    .sort_values("val_accuracy", ascending=False)
    .round(4)
)
initial_comparison

,train_accuracy,val_accuracy,val_std,gap
model,,,,
logistic_regression,1.0000,0.9791,0.0277,0.0209
knn,0.9771,0.9507,0.0353,0.0264
decision_tree,1.0000,0.9022,0.0586,0.0978


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 2, 5],
    },
    "logistic_regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 15],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipeline in models.items():
    search = GridSearchCV(
        pipeline, param_grids[name],
        scoring="accuracy", cv=cv, return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    best = search.best_index_
    results = search.cv_results_
    rows.append({
        "model": name,
        "best_params": search.best_params_,
        "cv_accuracy": search.best_score_,
        "cv_std": results["std_test_score"][best],
        "train_accuracy": results["mean_train_score"][best],
    })

# Empates en accuracy se resuelven a favor del modelo más estable (menor std)
tuned_comparison = (
    pd.DataFrame(rows).set_index("model")
    .sort_values(["cv_accuracy", "cv_std"], ascending=[False, True])
)
tuned_comparison.round(4)

,best_params,cv_accuracy,cv_std,train_accuracy
model,,,,
logistic_regression,{'model__C': 1},0.9791,0.0277,1.0000
knn,"{'model__n_neighbors': 7, 'model__weights': 'u...",0.9579,0.0405,0.9648
decision_tree,"{'model__criterion': 'gini', 'model__max_depth...",0.9025,0.0762,0.9859


## 6. Selección y persistencia del campeón

In [8]:
champion_name = tuned_comparison.index[0]
# GridSearchCV (refit=True) ya reentrenó el mejor pipeline con todo train
champion = searches[champion_name].best_estimator_
print("Campeón:", champion_name, searches[champion_name].best_params_)

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy": float(tuned_comparison.loc[champion_name, "cv_accuracy"]),
    "cv_std": float(tuned_comparison.loc[champion_name, "cv_std"]),
    "cv_folds": cv.get_n_splits(),
    "scoring": "accuracy",
    "random_state": RANDOM_STATE,
    "n_train": len(X_train),
    "features": feature_columns,
    "candidates": {
        name: float(score) for name, score in tuned_comparison["cv_accuracy"].items()
    },
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2), encoding="utf-8"
)

cv_results = pd.concat(
    [pd.DataFrame(search.cv_results_).assign(model=name) for name, search in searches.items()],
    ignore_index=True,
)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print("Artefactos guardados.")

Campeón: logistic_regression {'model__C': 1}
Artefactos guardados.


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> **Campeón: regresión logística (`C = 1`)**, con accuracy de CV de **0.979 ± 0.028**.
>
> - **Desempeño:** es el mejor de los tres (KNN 0.958, árbol 0.902).
> - **Variabilidad:** tiene la menor desviación estándar entre folds (0.028 frente a 0.041 de KNN y 0.076 del árbol), es decir, es el más estable.
> - **Sobreajuste:** el accuracy de train es 1.0, pero la brecha con validación es de solo ~2 puntos. El árbol muestra la mayor brecha (0.986 frente a 0.902), señal de que memoriza el entrenamiento.
> - **Complejidad:** es un modelo lineal con 13 × 3 coeficientes, rápido de entrenar y de predecir. KNN necesita guardar todo el dataset para predecir.
> - **Interpretabilidad:** los coeficientes (sobre variables estandarizadas) indican cuánto empuja cada variable hacia cada clase. El árbol sería más visual, pero aquí rinde peor y es menos estable.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.